# RI-13：结果关联的合成验证

只使用内存中的虚构名单和评分；不读取项目登记、真实名单或评分，不启动查看器。
依次验证：一对多、多对一、来源身份、未评分与无来源、非法列名拒绝。
本 notebook 使用项目 `.venv` 运行核心代码；Jupyter 自己的 kernel 不需要安装 EasyQC 依赖。


In [1]:
from pathlib import Path
import subprocess
import textwrap

candidates = [candidate for parent in (Path.cwd(), *Path.cwd().parents)
              for candidate in (parent, parent / "easyqc")]
repo = next(candidate for candidate in candidates if (candidate / "core/result_association.py").is_file())
python = repo / ".venv" / ("Scripts/python.exe" if __import__("os").name == "nt" else "bin/python")
assert python.is_file(), "先安装项目开发环境 .venv"
print(subprocess.check_output([str(python), "--version"], text=True).strip())

fixture = textwrap.dedent("""
    import pandas as pd
    from core.result_association import project_associations
    from models.result_association import ResultAssociationRule
    from models.rating import Rating
    from models.table_view_state import FilterCondition, FilterGroup, FilterExpression, filter_expression_to_json_object
    master = pd.DataFrame({
        "easyqcid": ["S01_anat", "S01_rest01", "S01_rest02", "S02_rest01"],
        "subses": ["01", "01", "01", "02"],
        "kind": ["anat", "func", "func", "func"],
    })
    def where(kind):
        return filter_expression_to_json_object(FilterExpression(groups=(
            FilterGroup("g", "all", (FilterCondition("kind", "==", kind, "c"),)),
        )))
    ratings = (
        Rating("AnatQC", "alice", "S01_anat", {"1": "Good"}, {}),
        Rating("RestQC", "bob", "S01_rest01", {"1": "Good"}, {}),
        Rating("RestQC", "bob", "S01_rest02", {"1": "Poor"}, {}),
    )
    def rule(module, rater, source, target, output):
        return ResultAssociationRule(output, output, module, rater, "subses", "subses",
            (("score1", output),), where(source), where(target))
""")
def run_probe(source):
    result = subprocess.run([str(python), "-c", fixture + "\n" + textwrap.dedent(source)],
                            cwd=repo, capture_output=True, text=True)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    result.check_returncode()


Python 3.10.17


## 1. 结构像评分共享给多个功能 run
输入总名单四行；投影仍为四行，两个 S01 功能 run 使用同一结构像来源。


In [2]:
run_probe("""
    print(master.to_string(index=False))
    projection = project_associations(master, ratings,
        (rule("AnatQC", "alice", "anat", "func", "shared_anat"),))
    print("\\n投影：")
    print(projection.columns.to_string(index=False))
    assert len(projection.columns) == len(master)
    assert projection.columns.loc[1, "shared_anat"] == projection.columns.loc[2, "shared_anat"]
""")


  easyqcid subses kind
  S01_anat     01 anat
S01_rest01     01 func
S01_rest02     01 func
S02_rest01     02 func

投影：
  easyqcid    shared_anat
  S01_anat               
S01_rest01 S01_anat: Good
S01_rest02 S01_anat: Good
S02_rest01               


## 2. 多个功能 run 的评分映射回一个结构像
保留每个来源的值，不求平均、不复制评分，并保留精确跳转身份。


In [3]:
run_probe("""
    projection = project_associations(master, ratings,
        (rule("RestQC", "bob", "func", "anat", "linked_rest"),))
    print(projection.columns.to_string(index=False))
    sources = projection.sources_by_easyqcid["S01_anat"]
    print([(s.easyqcid, s.module_name, s.rater) for s in sources])
    assert len(sources) == 2
    assert "Good" in projection.columns.loc[0, "linked_rest"]
    assert "Poor" in projection.columns.loc[0, "linked_rest"]
""")


  easyqcid                        linked_rest
  S01_anat S01_rest01: Good; S01_rest02: Poor
S01_rest01                                   
S01_rest02                                   
S02_rest01                                   
[('S01_rest01', 'RestQC', 'bob'), ('S01_rest02', 'RestQC', 'bob')]


## 3. 找到但未评分，不同于没有匹配来源


In [4]:
run_probe("""
    projection = project_associations(master, (),
        (rule("AnatQC", "alice", "anat", "func", "shared_anat"),))
    print(projection.columns.to_string(index=False))
    assert projection.columns.loc[1, "shared_anat"] == "S01_anat: —"
    assert projection.columns.loc[3, "shared_anat"] == ""
    print("未评分与无来源已区分")
""")


  easyqcid shared_anat
  S01_anat            
S01_rest01 S01_anat: —
S01_rest02 S01_anat: —
S02_rest01            
未评分与无来源已区分


## 4. 非法输出列名显式拒绝


In [5]:
run_probe("""
    try:
        rule("AnatQC", "alice", "anat", "func", "OtherQC.bob.score1")
    except ValueError as error:
        print(type(error).__name__ + ": " + str(error))
    else:
        raise AssertionError("不能占用自然结果列命名空间")
""")


ValueError: reserved output name: 'OtherQC.bob.score1'


## 验证边界

以上断言只验证合成数据上的核心关联，不代表真实项目、原生窗口、操作系统剪贴板或外部查看器已验证。
GUI 生命周期、只读模式、保存失败恢复和评分文件未改写由自动化测试覆盖，见 RI-13 实施/验收记录。
